# 00 — Prepare private model dataset (Kaggle, Internet ON)

Notebook **standalone**: không cần repo, không cần dataset code. Chạy hai version, mỗi version chọn một `MODEL_KEY`. Gemma 3 cần chấp nhận license trên Hugging Face và Kaggle Secret `HF_TOKEN`.


In [ ]:
# ===== CONFIG DUY NHẤT CẦN ĐỔI =====
MODEL_KEY = 'qwen25_7b'  # qwen25_7b | gemma3_4b

MODEL_REGISTRY = {
    'qwen25_7b': {
        'model_id': 'Qwen/Qwen2.5-7B-Instruct',
        'loader': 'causal_lm',
        'gated': False,
    },
    'gemma3_4b': {
        'model_id': 'google/gemma-3-4b-it',
        'loader': 'gemma3_conditional',
        'gated': True,
    },
}
DOWNLOAD_WHEELS = True
HASH_LARGE_FILES = True
assert MODEL_KEY in MODEL_REGISTRY
MODEL_SPEC = MODEL_REGISTRY[MODEL_KEY]
MODEL_SPEC


In [ ]:
# Internet phải ON ở notebook 00. Phiên bản >=4.51 cần cho Gemma 3.
%pip install -q "transformers>=4.51,<5" "accelerate>=0.34,<2" "bitsandbytes>=0.43,<1" "huggingface-hub>=0.27,<2" "safetensors>=0.4,<1" "tokenizers>=0.20,<1" "sentencepiece>=0.2,<1"


In [ ]:
# AUTO-EMBEDDED: standalone runtime; no repo import is required.

import hashlib
import json
import os
import platform
import tempfile
from pathlib import Path
from typing import Any, Iterable, Iterator


KIT_SCHEMA_VERSION = "local-llm-ablation-kit-v2"
BUNDLE_SCHEMA_VERSION = "model-agnostic-context-bundle-v2"
PREDICTION_SCHEMA_VERSION = "local-llm-predictions-v2"
JUDGE_SCHEMA_VERSION = "local-llm-gemini-judge-v2"


def canonical_json(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"), default=json_default)


def json_default(value: Any) -> Any:
    if hasattr(value, "model_dump"):
        return value.model_dump(mode="json")
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, set):
        return sorted(value)
    raise TypeError(f"Object of type {type(value).__name__} is not JSON serializable")


def sha256_text(value: str) -> str:
    return hashlib.sha256(value.encode("utf-8")).hexdigest()


def sha256_file(path: Path, *, block_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(block_size):
            digest.update(block)
    return digest.hexdigest()


def stable_pair_id(*parts: str) -> str:
    raw = "::".join(str(part) for part in parts)
    return sha256_text(raw)[:24]


def shard_for_pair(pair_id: str, num_shards: int) -> int:
    if num_shards < 1:
        raise ValueError("num_shards must be positive")
    return int(sha256_text(pair_id)[:16], 16) % num_shards


def iter_json_records(path: Path) -> Iterator[dict[str, Any]]:
    """Read JSONL, pretty multi-line concatenated JSON, or a JSON array."""
    content = path.read_text(encoding="utf-8")
    decoder = json.JSONDecoder()
    offset = 0
    while offset < len(content):
        while offset < len(content) and (content[offset].isspace() or content[offset] == ","):
            offset += 1
        if offset >= len(content):
            break
        if content[offset] == "[":
            payload = json.loads(content)
            if not isinstance(payload, list):
                raise ValueError(f"Expected JSON array in {path}")
            for record in payload:
                if isinstance(record, dict):
                    yield record
            return
        payload, next_offset = decoder.raw_decode(content, offset)
        if not isinstance(payload, dict):
            raise ValueError(f"Expected JSON object at character {offset} in {path}")
        yield payload
        offset = next_offset


def load_jsonl_map(path: Path, key: str) -> dict[str, dict[str, Any]]:
    result: dict[str, dict[str, Any]] = {}
    if not path.exists():
        return result
    for record in iter_json_records(path):
        value = str(record.get(key) or "")
        if not value:
            raise ValueError(f"Record in {path} is missing key {key}")
        if value in result and canonical_json(result[value]) != canonical_json(record):
            raise ValueError(f"Conflicting duplicate {key}={value} in {path}")
        result[value] = record
    return result


def append_jsonl(path: Path, record: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8", newline="\n") as handle:
        handle.write(json.dumps(record, ensure_ascii=False, sort_keys=True, default=json_default))
        handle.write("\n")
        handle.flush()
        os.fsync(handle.fileno())


def atomic_write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
    try:
        with os.fdopen(fd, "w", encoding="utf-8", newline="\n") as handle:
            json.dump(payload, handle, ensure_ascii=False, indent=2, sort_keys=True, default=json_default)
            handle.write("\n")
        os.replace(temporary, path)
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)


def write_jsonl_atomic(path: Path, records: Iterable[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temporary = tempfile.mkstemp(prefix=f".{path.name}.", suffix=".tmp", dir=path.parent)
    try:
        with os.fdopen(fd, "w", encoding="utf-8", newline="\n") as handle:
            for record in records:
                handle.write(json.dumps(record, ensure_ascii=False, sort_keys=True, default=json_default))
                handle.write("\n")
        os.replace(temporary, path)
    finally:
        if os.path.exists(temporary):
            os.unlink(temporary)


def resolve_directory(
    explicit: str | Path | None,
    *,
    marker: str,
    search_roots: Iterable[str | Path] = ("/kaggle/input", "."),
) -> Path:
    if explicit:
        candidate = Path(explicit).expanduser().resolve()
        if not (candidate / marker).exists():
            raise FileNotFoundError(f"{candidate} does not contain {marker}")
        return candidate
    matches: list[Path] = []
    for root in search_roots:
        root_path = Path(root)
        if not root_path.exists():
            continue
        matches.extend(path.parent.resolve() for path in root_path.rglob(marker))
    unique = sorted(set(matches))
    if len(unique) != 1:
        raise FileNotFoundError(f"Expected exactly one directory containing {marker}; found {unique}")
    return unique[0]


def environment_summary() -> dict[str, Any]:
    summary: dict[str, Any] = {
        "python": platform.python_version(),
        "platform": platform.platform(),
    }
    for package in ("torch", "transformers", "accelerate", "bitsandbytes", "safetensors"):
        try:
            module = __import__(package)
            summary[package] = str(getattr(module, "__version__", "unknown"))
        except Exception:
            summary[package] = None
    try:
        import torch

        summary["cuda_available"] = torch.cuda.is_available()
        summary["cuda_device_count"] = torch.cuda.device_count()
        summary["cuda_devices"] = [torch.cuda.get_device_name(index) for index in range(torch.cuda.device_count())]
    except Exception:
        summary["cuda_available"] = False
        summary["cuda_device_count"] = 0
        summary["cuda_devices"] = []
    return summary


def assert_no_secret_keys(payload: Any) -> None:
    forbidden = {"api_key", "gemini_api_key", "hf_token", "token", "password", "neo4j_password"}

    def walk(value: Any, path: str) -> None:
        if isinstance(value, dict):
            for key, child in value.items():
                normalized = str(key).strip().lower()
                if normalized in forbidden and child not in (None, "", False):
                    raise ValueError(f"Secret-like value found at {path}.{key}; never persist secrets in artifacts")
                walk(child, f"{path}.{key}")
        elif isinstance(value, list):
            for index, child in enumerate(value):
                walk(child, f"{path}[{index}]")

    walk(payload, "root")


import json
import os
import shutil
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any



DEFAULT_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
DEFAULT_WHEEL_PACKAGES = [
    "transformers",
    "accelerate",
    "bitsandbytes",
    "safetensors",
    "huggingface-hub",
    "tokenizers",
    "sentencepiece",
]


def prepare_model_assets(config: dict[str, Any]) -> dict[str, Any]:
    """Download a pinned HF snapshot and optional wheels for an offline Kaggle run."""
    from huggingface_hub import HfApi, snapshot_download

    model_id = str(config.get("model_id") or DEFAULT_MODEL_ID)
    model_key = str(config.get("model_key") or model_id.rsplit("/", 1)[-1]).strip()
    loader = str(config.get("loader") or "causal_lm").strip()
    requested_revision = str(config.get("revision") or "main")
    output_root = Path(config.get("output_root") or f"/kaggle/working/{model_key}_offline_dataset").resolve()
    model_dir = output_root / "model"
    wheelhouse = output_root / "wheelhouse"
    output_root.mkdir(parents=True, exist_ok=True)

    hf_token = config.get("hf_token") or os.getenv("HF_TOKEN") or None
    info = HfApi(token=hf_token).model_info(model_id, revision=requested_revision)
    resolved_revision = str(info.sha)
    snapshot_path = snapshot_download(
        repo_id=model_id,
        revision=resolved_revision,
        local_dir=model_dir,
        token=hf_token,
        allow_patterns=[
            "*.json",
            "*.model",
            "*.py",
            "*.safetensors",
            "*.txt",
            "*.jinja",
            "tokenizer*",
            "vocab*",
            "merges*",
        ],
        ignore_patterns=["*.bin", "*.h5", "*.msgpack", "*.onnx"],
    )
    # snapshot_download may create local transfer metadata. It is unnecessary for
    # inference and should not be published with a gated model dataset.
    shutil.rmtree(model_dir / ".cache", ignore_errors=True)

    if bool(config.get("download_wheels", True)):
        wheelhouse.mkdir(parents=True, exist_ok=True)
        packages = list(config.get("wheel_packages") or DEFAULT_WHEEL_PACKAGES)
        for package in packages:
            module_name = package.replace("-", "_")
            try:
                module = __import__(module_name)
                version = str(getattr(module, "__version__"))
            except Exception:
                continue
            subprocess.run(
                [
                    sys.executable,
                    "-m",
                    "pip",
                    "download",
                    "--no-deps",
                    "--dest",
                    str(wheelhouse),
                    f"{package}=={version}",
                ],
                check=True,
            )

    files: list[dict[str, Any]] = []
    hash_large_files = bool(config.get("hash_large_files", True))
    for path in sorted(file for file in output_root.rglob("*") if file.is_file()):
        size = path.stat().st_size
        checksum = sha256_file(path) if hash_large_files or size < 256 * 1024 * 1024 else None
        files.append(
            {
                "path": path.relative_to(output_root).as_posix(),
                "size_bytes": size,
                "sha256": checksum,
            }
        )

    manifest = {
        "schema_version": KIT_SCHEMA_VERSION,
        "asset_type": "offline_huggingface_model",
        "created_at": datetime.now(timezone.utc).isoformat(),
        "model_id": model_id,
        "model_key": model_key,
        "loader": loader,
        "requested_revision": requested_revision,
        "resolved_revision": resolved_revision,
        "snapshot_path": str(snapshot_path),
        "model_subdir": "model",
        "wheelhouse_subdir": "wheelhouse" if wheelhouse.exists() else None,
        "environment": environment_summary(),
        "file_count": len(files),
        "total_size_bytes": sum(int(item["size_bytes"]) for item in files),
        "files": files,
    }
    atomic_write_json(output_root / "asset_manifest.json", manifest)

    config_path = model_dir / "config.json"
    tokenizer_path = model_dir / "tokenizer_config.json"
    if not config_path.exists() or not tokenizer_path.exists():
        raise RuntimeError("Downloaded model is incomplete: config.json/tokenizer_config.json missing")
    if not list(model_dir.glob("*.safetensors")):
        raise RuntimeError("Downloaded model is incomplete: no safetensors weights found")
    return manifest


In [ ]:
hf_token = None
if MODEL_SPEC['gated']:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret('HF_TOKEN')
    assert hf_token, 'Thiếu Kaggle Secret HF_TOKEN hoặc chưa được cấp quyền Gemma'

MODEL_CONFIG = {
    'model_key': MODEL_KEY,
    'model_id': MODEL_SPEC['model_id'],
    'loader': MODEL_SPEC['loader'],
    'revision': 'main',
    'hf_token': hf_token,
    'download_wheels': DOWNLOAD_WHEELS,
    'hash_large_files': HASH_LARGE_FILES,
    'output_root': f'/kaggle/working/{MODEL_KEY}_offline_dataset',
}
print({key: ('<set>' if key == 'hf_token' and value else value) for key, value in MODEL_CONFIG.items()})
manifest = prepare_model_assets(MODEL_CONFIG)


In [ ]:
from pathlib import Path
output_root = Path(MODEL_CONFIG['output_root'])
assert (output_root / 'asset_manifest.json').exists()
assert (output_root / 'model' / 'config.json').exists()
assert list((output_root / 'model').glob('*.safetensors'))
assert manifest['model_id'] == MODEL_SPEC['model_id']
assert manifest['loader'] == MODEL_SPEC['loader']
print({
    'PASS': True,
    'output_root': str(output_root),
    'model_id': manifest['model_id'],
    'revision': manifest['resolved_revision'],
    'size_GB': round(manifest['total_size_bytes'] / 1e9, 2),
})


## Sau khi chạy

Save Version, sau đó tạo **private Kaggle Dataset** từ toàn bộ output folder. Không tải hoặc chia sẻ `HF_TOKEN`. Xem `KAGGLE_DATASET_GUIDE.md` trong repo để biết từng bước.
